## 1. Import Library

In [1]:
import pandas as pd 
import numpy as np

## 2. Data Pucang Anom

In [2]:
data_pucang = pd.read_csv("../data/raw/pasar/data_pucang_anom_kalender.csv", parse_dates=['tanggal'])
data_pucang.head()

,tanggal,komoditas_id,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan
0,2020-01-02,2,Beras Premium,BERAS,kg,12500,12500,Kamis,0,0,NaN,0,0
1,2020-01-02,4,Beras Medium,BERAS,kg,9000,9000,Kamis,0,0,NaN,0,0
2,2020-01-02,7,Gula Kristal Putih,GULA,kg,13000,13000,Kamis,0,0,NaN,0,0
3,2020-01-02,10,Minyak Goreng Curah,MINYAK GORENG,kg,12500,12500,Kamis,0,0,NaN,0,0
4,2020-01-02,92,Minyak Goreng Kemasan Premium,MINYAK GORENG,1 liter,12500,12500,Kamis,0,0,NaN,0,0


In [3]:
data_pucang.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102270 entries, 0 to 102269
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   tanggal            102270 non-null  datetime64[ns]
 1   komoditas_id       102270 non-null  int64         
 2   komoditas          102270 non-null  object        
 3   grup               102270 non-null  object        
 4   satuan             102270 non-null  object        
 5   harga_kemarin      102270 non-null  int64         
 6   harga              102270 non-null  int64         
 7   hari_nama          102270 non-null  object        
 8   is_weekend         102270 non-null  int64         
 9   is_libur_nasional  102270 non-null  int64         
 10  nama_libur         6930 non-null    object        
 11  is_ramadan         102270 non-null  int64         
 12  is_pra_ramadan     102270 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(5)
me

In [4]:
drop_data_pucang = ['Minyak Goreng Kemasan Sederhana', 'Susu Bubuk Merk Bendera (Instant)', 'Susu Bubuk Merk Indomilk (Instant)', 'Kedelai Lokal', 'Bata']

data_pucang['harga'] = data_pucang['harga'].replace(0, np.nan)
data_pucang['harga_kemarin'] = data_pucang['harga_kemarin'].replace(0, np.nan)

data_pucang_clean = data_pucang[~data_pucang['komoditas'].isin(drop_data_pucang)].copy()
data_pucang_clean.loc[data_pucang_clean['komoditas'] == 'Halus', 'komoditas'] = 'Garam Beryodium Halus'

full_tanggal = pd.date_range(data_pucang_clean['tanggal'].min(), data_pucang_clean['tanggal'].max(), freq='D')
grid = pd.MultiIndex.from_product([data_pucang_clean['komoditas_id'].unique(), full_tanggal],
                                  names=['komoditas_id', 'tanggal']).to_frame(index=False)

data_pucang_clean = grid.merge(data_pucang_clean, on=['komoditas_id', 'tanggal'], how='left')
for kolom in ['komoditas', 'grup', 'satuan']:
    data_pucang_clean[kolom] = data_pucang_clean.groupby('komoditas_id')[kolom].ffill().bfill()

hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
data_pucang_clean['hari_nama'] = data_pucang_clean['tanggal'].dt.weekday.map(lambda i: hari[i])
data_pucang_clean['is_weekend'] = (data_pucang_clean['tanggal'].dt.weekday >= 5).astype(int)

data_pucang_clean = data_pucang_clean.sort_values(['komoditas_id', 'tanggal']).reset_index(drop=True)
data_pucang_clean['imputed'] = data_pucang_clean['harga'].isna()
data_pucang_clean['harga'] = data_pucang_clean.groupby('komoditas_id')['harga'].transform(lambda s: s.ffill(limit=3).interpolate(limit=7))
data_pucang_clean['imputed'] = data_pucang_clean['imputed'] & data_pucang_clean['harga'].notna()

In [5]:
data_pucang_clean.head()

,komoditas_id,tanggal,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan,imputed
0,2,2020-01-01,Beras Premium,BERAS,kg,12500.0,12500.0,Rabu,0,1,New Year's Day,0,0,False
1,2,2020-01-02,Beras Premium,BERAS,kg,12500.0,12500.0,Kamis,0,0,NaN,0,0,False
2,2,2020-01-03,Beras Premium,BERAS,kg,12500.0,12500.0,Jumat,0,0,NaN,0,0,False
3,2,2020-01-04,Beras Premium,BERAS,kg,12500.0,12500.0,Sabtu,1,0,NaN,0,0,False
4,2,2020-01-05,Beras Premium,BERAS,kg,12500.0,12500.0,Minggu,1,0,NaN,0,0,False


In [6]:
data_pucang_clean.info()
data_pucang_clean[data_pucang_clean['harga'].isna()].groupby('komoditas')['tanggal'].count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90095 entries, 0 to 90094
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   komoditas_id       90095 non-null  int64         
 1   tanggal            90095 non-null  datetime64[ns]
 2   komoditas          90095 non-null  object        
 3   grup               90095 non-null  object        
 4   satuan             90095 non-null  object        
 5   harga_kemarin      88936 non-null  float64       
 6   harga              89002 non-null  float64       
 7   hari_nama          90095 non-null  object        
 8   is_weekend         90095 non-null  int64         
 9   is_libur_nasional  90095 non-null  int64         
 10  nama_libur         6105 non-null   object        
 11  is_ramadan         90095 non-null  int64         
 12  is_pra_ramadan     90095 non-null  int64         
 13  imputed            90095 non-null  bool          
dtypes: boo

komoditas
Minyak Goreng Kemasan Premium       51
Minyak Goreng MINYAKITA           1004
Susu Kental Manis Merk Bendera      38
Name: tanggal, dtype: int64

## 3. Data Tambah Rejo

In [7]:
data_tambahrejo = pd.read_csv("../data/raw/pasar/data_tambahrejo_kalender.csv", parse_dates=['tanggal'])
data_tambahrejo.head()

,tanggal,komoditas_id,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan
0,2020-01-06,2,Beras Premium,BERAS,kg,12500,12500,Senin,0,0,NaN,0,0
1,2020-01-06,4,Beras Medium,BERAS,kg,9000,9000,Senin,0,0,NaN,0,0
2,2020-01-06,7,Gula Kristal Putih,GULA,kg,13000,13000,Senin,0,0,NaN,0,0
3,2020-01-06,10,Minyak Goreng Curah,MINYAK GORENG,kg,12500,12500,Senin,0,0,NaN,0,0
4,2020-01-06,92,Minyak Goreng Kemasan Premium,MINYAK GORENG,1 liter,12000,12000,Senin,0,0,NaN,0,0


In [8]:
data_tambahrejo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102270 entries, 0 to 102269
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   tanggal            102270 non-null  datetime64[ns]
 1   komoditas_id       102270 non-null  int64         
 2   komoditas          102270 non-null  object        
 3   grup               102270 non-null  object        
 4   satuan             102270 non-null  object        
 5   harga_kemarin      102270 non-null  int64         
 6   harga              102270 non-null  int64         
 7   hari_nama          102270 non-null  object        
 8   is_weekend         102270 non-null  int64         
 9   is_libur_nasional  102270 non-null  int64         
 10  nama_libur         6930 non-null    object        
 11  is_ramadan         102270 non-null  int64         
 12  is_pra_ramadan     102270 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(5)
me

In [9]:
drop_data_tambahrejo = ['Minyak Goreng Kemasan Sederhana', 'Susu Bubuk Merk Bendera (Instant)', 'Susu Bubuk Merk Indomilk (Instant)', 'Kedelai Lokal', 'Bata',
                    'Daging Ayam Kampung', 'Ikan Tuna', 'Ikan Cakalang', 'Ikan Asin Teri', 'Ikan Kembung']

data_tambahrejo['harga'] = data_tambahrejo['harga'].replace(0, np.nan)
data_tambahrejo['harga_kemarin'] = data_tambahrejo['harga_kemarin'].replace(0, np.nan)

data_tambahrejo_clean = data_tambahrejo[~data_tambahrejo['komoditas'].isin(drop_data_tambahrejo)].copy()
data_tambahrejo_clean.loc[data_tambahrejo_clean['komoditas'] == 'Halus', 'komoditas'] = 'Garam Beryodium Halus'

full_tanggal = pd.date_range(data_tambahrejo_clean['tanggal'].min(), data_tambahrejo_clean['tanggal'].max(), freq='D')
grid = pd.MultiIndex.from_product([data_tambahrejo_clean['komoditas_id'].unique(), full_tanggal],
                                  names=['komoditas_id', 'tanggal']).to_frame(index=False)

data_tambahrejo_clean = grid.merge(data_tambahrejo_clean, on=['komoditas_id', 'tanggal'], how='left')
for kolom in ['komoditas', 'grup', 'satuan']:
    data_tambahrejo_clean[kolom] = data_tambahrejo_clean.groupby('komoditas_id')[kolom].ffill().bfill()

hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
data_tambahrejo_clean['hari_nama'] = data_tambahrejo_clean['tanggal'].dt.weekday.map(lambda i: hari[i])
data_tambahrejo_clean['is_weekend'] = (data_tambahrejo_clean['tanggal'].dt.weekday >= 5).astype(int)

data_tambahrejo_clean = data_tambahrejo_clean.sort_values(['komoditas_id', 'tanggal']).reset_index(drop=True)
data_tambahrejo_clean['imputed'] = data_tambahrejo_clean['harga'].isna()
data_tambahrejo_clean['harga'] = data_tambahrejo_clean.groupby('komoditas_id')['harga'].transform(lambda s: s.ffill(limit=3).interpolate(limit=7))
data_tambahrejo_clean['imputed'] = data_tambahrejo_clean['imputed'] & data_tambahrejo_clean['harga'].notna()

In [10]:
data_tambahrejo_clean.head()

,komoditas_id,tanggal,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan,imputed
0,2,2020-01-01,Beras Premium,BERAS,kg,12500.0,12500.0,Rabu,0,1,New Year's Day,0,0,False
1,2,2020-01-02,Beras Premium,BERAS,kg,12500.0,12500.0,Kamis,0,0,NaN,0,0,False
2,2,2020-01-03,Beras Premium,BERAS,kg,12500.0,12500.0,Jumat,0,0,NaN,0,0,False
3,2,2020-01-04,Beras Premium,BERAS,kg,12500.0,12500.0,Sabtu,1,0,NaN,0,0,False
4,2,2020-01-05,Beras Premium,BERAS,kg,12500.0,12500.0,Minggu,1,0,NaN,0,0,False


In [11]:
data_tambahrejo_clean.info()
data_tambahrejo_clean[data_tambahrejo_clean['harga'].isna()].groupby('komoditas')['tanggal'].count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77920 entries, 0 to 77919
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   komoditas_id       77920 non-null  int64         
 1   tanggal            77920 non-null  datetime64[ns]
 2   komoditas          77920 non-null  object        
 3   grup               77920 non-null  object        
 4   satuan             77920 non-null  object        
 5   harga_kemarin      76699 non-null  float64       
 6   harga              76850 non-null  float64       
 7   hari_nama          77920 non-null  object        
 8   is_weekend         77920 non-null  int64         
 9   is_libur_nasional  77920 non-null  int64         
 10  nama_libur         5280 non-null   object        
 11  is_ramadan         77920 non-null  int64         
 12  is_pra_ramadan     77920 non-null  int64         
 13  imputed            77920 non-null  bool          
dtypes: boo

komoditas
Bawang Putih Sinco/Honan           15
Minyak Goreng Kemasan Premium      47
Minyak Goreng MINYAKITA          1008
Name: tanggal, dtype: int64

## 4. Data Wonokromo

In [12]:
data_wonokromo = pd.read_csv("../data/raw/pasar/data_wonokromo_kalender.csv", parse_dates=['tanggal'])
data_wonokromo.head()

,tanggal,komoditas_id,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan
0,2020-01-05,2,Beras Premium,BERAS,kg,12800,12800,Minggu,1,0,NaN,0,0
1,2020-01-05,4,Beras Medium,BERAS,kg,9800,9800,Minggu,1,0,NaN,0,0
2,2020-01-05,7,Gula Kristal Putih,GULA,kg,12000,12000,Minggu,1,0,NaN,0,0
3,2020-01-05,10,Minyak Goreng Curah,MINYAK GORENG,kg,10500,10500,Minggu,1,0,NaN,0,0
4,2020-01-05,92,Minyak Goreng Kemasan Premium,MINYAK GORENG,1 liter,12000,12000,Minggu,1,0,NaN,0,0


In [13]:
data_wonokromo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102270 entries, 0 to 102269
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   tanggal            102270 non-null  datetime64[ns]
 1   komoditas_id       102270 non-null  int64         
 2   komoditas          102270 non-null  object        
 3   grup               102270 non-null  object        
 4   satuan             102270 non-null  object        
 5   harga_kemarin      102270 non-null  int64         
 6   harga              102270 non-null  int64         
 7   hari_nama          102270 non-null  object        
 8   is_weekend         102270 non-null  int64         
 9   is_libur_nasional  102270 non-null  int64         
 10  nama_libur         6930 non-null    object        
 11  is_ramadan         102270 non-null  int64         
 12  is_pra_ramadan     102270 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(5)
me

In [14]:
drop_data_wonokromo = ['Minyak Goreng Kemasan Sederhana', 'Susu Bubuk Merk Bendera (Instant)', 'Susu Bubuk Merk Indomilk (Instant)', 'Kedelai Lokal', 'Bata',
                       'Ikan Kembung', 'Ikan Tongkol', 'Ikan Asin Teri', 'Ikan Bandeng', 'Minyak Goreng MINYAKITA']

data_wonokromo['harga'] = data_wonokromo['harga'].replace(0, np.nan)
data_wonokromo['harga_kemarin'] = data_wonokromo['harga_kemarin'].replace(0, np.nan)

data_wonokromo_clean = data_wonokromo[~data_wonokromo['komoditas'].isin(drop_data_wonokromo)].copy()
data_wonokromo_clean.loc[data_wonokromo_clean['komoditas'] == 'Halus', 'komoditas'] = 'Garam Beryodium Halus'

full_tanggal = pd.date_range(data_wonokromo_clean['tanggal'].min(), data_wonokromo_clean['tanggal'].max(), freq='D')
grid = pd.MultiIndex.from_product([data_wonokromo_clean['komoditas_id'].unique(), full_tanggal],
                                  names=['komoditas_id', 'tanggal']).to_frame(index=False)

data_wonokromo_clean = grid.merge(data_wonokromo_clean, on=['komoditas_id', 'tanggal'], how='left')
for kolom in ['komoditas', 'grup', 'satuan']:
    data_wonokromo_clean[kolom] = data_wonokromo_clean.groupby('komoditas_id')[kolom].ffill().bfill()

hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
data_wonokromo_clean['hari_nama'] = data_wonokromo_clean['tanggal'].dt.weekday.map(lambda i: hari[i])
data_wonokromo_clean['is_weekend'] = (data_wonokromo_clean['tanggal'].dt.weekday >= 5).astype(int)

data_wonokromo_clean = data_wonokromo_clean.sort_values(['komoditas_id', 'tanggal']).reset_index(drop=True)
data_wonokromo_clean['imputed'] = data_wonokromo_clean['harga'].isna()
data_wonokromo_clean['harga'] = data_wonokromo_clean.groupby('komoditas_id')['harga'].transform(lambda s: s.ffill(limit=3).interpolate(limit=7))
data_wonokromo_clean['imputed'] = data_wonokromo_clean['imputed'] & data_wonokromo_clean['harga'].notna()

In [15]:
data_wonokromo_clean.head()

,komoditas_id,tanggal,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan,imputed
0,2,2020-01-01,Beras Premium,BERAS,kg,12800.0,12800.0,Rabu,0,1,New Year's Day,0,0,False
1,2,2020-01-02,Beras Premium,BERAS,kg,12800.0,12800.0,Kamis,0,0,NaN,0,0,False
2,2,2020-01-03,Beras Premium,BERAS,kg,12800.0,12800.0,Jumat,0,0,NaN,0,0,False
3,2,2020-01-04,Beras Premium,BERAS,kg,12800.0,12800.0,Sabtu,1,0,NaN,0,0,False
4,2,2020-01-05,Beras Premium,BERAS,kg,12800.0,12800.0,Minggu,1,0,NaN,0,0,False


In [16]:
data_wonokromo_clean.info()
data_wonokromo_clean[data_wonokromo_clean['harga'].isna()].groupby('komoditas')['tanggal'].count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77920 entries, 0 to 77919
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   komoditas_id       77920 non-null  int64         
 1   tanggal            77920 non-null  datetime64[ns]
 2   komoditas          77920 non-null  object        
 3   grup               77920 non-null  object        
 4   satuan             77920 non-null  object        
 5   harga_kemarin      72937 non-null  float64       
 6   harga              72977 non-null  float64       
 7   hari_nama          77920 non-null  object        
 8   is_weekend         77920 non-null  int64         
 9   is_libur_nasional  77920 non-null  int64         
 10  nama_libur         5280 non-null   object        
 11  is_ramadan         77920 non-null  int64         
 12  is_pra_ramadan     77920 non-null  int64         
 13  imputed            77920 non-null  bool          
dtypes: boo

komoditas
Ikan Cakalang                      2435
Ikan Tuna                          2435
Minyak Goreng Kemasan Premium        63
Terigu Protein Sedang (Kemasan)      10
Name: tanggal, dtype: int64

## 5. Data Soponyono

In [17]:
data_soponyono = pd.read_csv('../data/raw/pasar/data_Soponyono_kalender.csv', parse_dates=['tanggal'])
data_soponyono.head()

,tanggal,komoditas_id,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan
0,2020-01-05,2,Beras Premium,BERAS,kg,0,0,Minggu,1,0,NaN,0,0
1,2020-01-05,4,Beras Medium,BERAS,kg,0,0,Minggu,1,0,NaN,0,0
2,2020-01-05,7,Gula Kristal Putih,GULA,kg,0,0,Minggu,1,0,NaN,0,0
3,2020-01-05,10,Minyak Goreng Curah,MINYAK GORENG,kg,0,0,Minggu,1,0,NaN,0,0
4,2020-01-05,92,Minyak Goreng Kemasan Premium,MINYAK GORENG,1 liter,0,0,Minggu,1,0,NaN,0,0


In [18]:
data_soponyono.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102270 entries, 0 to 102269
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   tanggal            102270 non-null  datetime64[ns]
 1   komoditas_id       102270 non-null  int64         
 2   komoditas          102270 non-null  object        
 3   grup               102270 non-null  object        
 4   satuan             102270 non-null  object        
 5   harga_kemarin      102270 non-null  int64         
 6   harga              102270 non-null  int64         
 7   hari_nama          102270 non-null  object        
 8   is_weekend         102270 non-null  int64         
 9   is_libur_nasional  102270 non-null  int64         
 10  nama_libur         6930 non-null    object        
 11  is_ramadan         102270 non-null  int64         
 12  is_pra_ramadan     102270 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(5)
me

In [19]:
drop_data_soponyono = ['Minyak Goreng Kemasan Sederhana', 'Susu Bubuk Merk Bendera (Instant)', 'Susu Bubuk Merk Indomilk (Instant)', 'Kedelai Lokal',
                       'Jagung Pipilan Kering', 'Minyak Goreng MINYAKITA', 'Bata']

data_soponyono = data_soponyono[data_soponyono['tanggal'] >= '2023-01-01']

data_soponyono['harga'] = data_soponyono['harga'].replace(0, np.nan)
data_soponyono['harga_kemarin'] = data_soponyono['harga_kemarin'].replace(0, np.nan)

data_soponyono_clean = data_soponyono[~data_soponyono['komoditas'].isin(drop_data_soponyono)].copy()
data_soponyono_clean.loc[data_soponyono_clean['komoditas'] == 'Halus', 'komoditas'] = 'Garam Beryodium Halus'

full_tanggal = pd.date_range('2023-01-01', data_soponyono_clean['tanggal'].max(), freq='D')
grid = pd.MultiIndex.from_product([data_soponyono_clean['komoditas_id'].unique(), full_tanggal],
                                  names=['komoditas_id', 'tanggal']).to_frame(index=False)

data_soponyono_clean = grid.merge(data_soponyono_clean, on=['komoditas_id', 'tanggal'], how='left')
for kolom in ['komoditas', 'grup', 'satuan']:
    data_soponyono_clean[kolom] = data_soponyono_clean.groupby('komoditas_id')[kolom].ffill().bfill()

hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
data_soponyono_clean['hari_nama'] = data_soponyono_clean['tanggal'].dt.weekday.map(lambda i: hari[i])
data_soponyono_clean['is_weekend'] = (data_soponyono_clean['tanggal'].dt.weekday >= 5).astype(int)

data_soponyono_clean = data_soponyono_clean.sort_values(['komoditas_id', 'tanggal']).reset_index(drop=True)
data_soponyono_clean['imputed'] = data_soponyono_clean['harga'].isna()
data_soponyono_clean['harga'] = data_soponyono_clean.groupby('komoditas_id')['harga'].transform(lambda s: s.ffill(limit=3).interpolate(limit=7))
data_soponyono_clean['imputed'] = data_soponyono_clean['imputed'] & data_soponyono_clean['harga'].notna()

mask_beras = data_soponyono_clean['komoditas'] == 'Beras Medium'
awal = data_soponyono_clean.loc[mask_beras & data_soponyono_clean['harga'].notna(), 'tanggal'].min()
data_soponyono_clean = data_soponyono_clean[~(mask_beras & (data_soponyono_clean['tanggal'] < awal))].reset_index(drop=True)

In [20]:
data_soponyono_clean.head()

,komoditas_id,tanggal,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan,imputed
0,2,2023-01-01,Beras Premium,BERAS,kg,12000.0,12000.0,Minggu,1,1,New Year's Day,0,0,False
1,2,2023-01-02,Beras Premium,BERAS,kg,12000.0,12000.0,Senin,0,0,NaN,0,0,False
2,2,2023-01-03,Beras Premium,BERAS,kg,12000.0,12000.0,Selasa,0,0,NaN,0,0,False
3,2,2023-01-04,Beras Premium,BERAS,kg,12000.0,12400.0,Rabu,0,0,NaN,0,0,False
4,2,2023-01-05,Beras Premium,BERAS,kg,12400.0,12400.0,Kamis,0,0,NaN,0,0,False


In [21]:
data_soponyono_clean.info()
data_soponyono_clean[data_soponyono_clean['harga'].isna()].groupby('komoditas')['tanggal'].count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46865 entries, 0 to 46864
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   komoditas_id       46865 non-null  int64         
 1   tanggal            46865 non-null  datetime64[ns]
 2   komoditas          46865 non-null  object        
 3   grup               46865 non-null  object        
 4   satuan             46865 non-null  object        
 5   harga_kemarin      46715 non-null  float64       
 6   harga              46755 non-null  float64       
 7   hari_nama          46865 non-null  object        
 8   is_weekend         46865 non-null  int64         
 9   is_libur_nasional  46865 non-null  int64         
 10  nama_libur         3745 non-null   object        
 11  is_ramadan         46865 non-null  int64         
 12  is_pra_ramadan     46865 non-null  int64         
 13  imputed            46865 non-null  bool          
dtypes: boo

komoditas
Beras Medium    110
Name: tanggal, dtype: int64

## 6. Data Keputran

In [22]:
data_keputran = pd.read_csv('../data/raw/pasar/data_keputran_kalender.csv', parse_dates=['tanggal'])
data_keputran.head()

,tanggal,komoditas_id,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan
0,2020-01-06,2,Beras Premium,BERAS,kg,11000,11000,Senin,0,0,NaN,0,0
1,2020-01-06,4,Beras Medium,BERAS,kg,0,0,Senin,0,0,NaN,0,0
2,2020-01-06,7,Gula Kristal Putih,GULA,kg,12500,12500,Senin,0,0,NaN,0,0
3,2020-01-06,10,Minyak Goreng Curah,MINYAK GORENG,kg,12000,13000,Senin,0,0,NaN,0,0
4,2020-01-06,92,Minyak Goreng Kemasan Premium,MINYAK GORENG,1 liter,0,0,Senin,0,0,NaN,0,0


In [23]:
data_keputran.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102270 entries, 0 to 102269
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   tanggal            102270 non-null  datetime64[ns]
 1   komoditas_id       102270 non-null  int64         
 2   komoditas          102270 non-null  object        
 3   grup               102270 non-null  object        
 4   satuan             102270 non-null  object        
 5   harga_kemarin      102270 non-null  int64         
 6   harga              102270 non-null  int64         
 7   hari_nama          102270 non-null  object        
 8   is_weekend         102270 non-null  int64         
 9   is_libur_nasional  102270 non-null  int64         
 10  nama_libur         6930 non-null    object        
 11  is_ramadan         102270 non-null  int64         
 12  is_pra_ramadan     102270 non-null  int64         
dtypes: datetime64[ns](1), int64(7), object(5)
me

In [24]:
drop_data_keputran = ['Minyak Goreng Kemasan Sederhana', 'Susu Bubuk Merk Bendera (Instant)', 'Susu Bubuk Merk Indomilk (Instant)', 'Kedelai Lokal',
                      'Ikan Tuna', 'Ikan Cakalang', 'Ikan Kembung', 'Ikan Asin Teri', 'Ikan Tongkol',
                      'KETELA POHON', 'Jagung Pipilan Kering', 'Minyak Goreng MINYAKITA', 'Cabe Merah Keriting', 'Bata']

data_keputran['harga'] = data_keputran['harga'].replace(0, np.nan)
data_keputran['harga_kemarin'] = data_keputran['harga_kemarin'].replace(0, np.nan)

data_keputran_clean = data_keputran[~data_keputran['komoditas'].isin(drop_data_keputran)].copy()
data_keputran_clean.loc[data_keputran_clean['komoditas'] == 'Halus', 'komoditas'] = 'Garam Beryodium Halus'

full_tanggal = pd.date_range(data_keputran_clean['tanggal'].min(), data_keputran_clean['tanggal'].max(), freq='D')
grid = pd.MultiIndex.from_product([data_keputran_clean['komoditas_id'].unique(), full_tanggal],
                                  names=['komoditas_id', 'tanggal']).to_frame(index=False)

data_keputran_clean = grid.merge(data_keputran_clean, on=['komoditas_id', 'tanggal'], how='left')
for kolom in ['komoditas', 'grup', 'satuan']:
    data_keputran_clean[kolom] = data_keputran_clean.groupby('komoditas_id')[kolom].ffill().bfill()

hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
data_keputran_clean['hari_nama'] = data_keputran_clean['tanggal'].dt.weekday.map(lambda i: hari[i])
data_keputran_clean['is_weekend'] = (data_keputran_clean['tanggal'].dt.weekday >= 5).astype(int)

data_keputran_clean = data_keputran_clean.sort_values(['komoditas_id', 'tanggal']).reset_index(drop=True)
data_keputran_clean['imputed'] = data_keputran_clean['harga'].isna()
data_keputran_clean['harga'] = data_keputran_clean.groupby('komoditas_id')['harga'].transform(lambda s: s.ffill(limit=3).interpolate(limit=7))
data_keputran_clean['imputed'] = data_keputran_clean['imputed'] & data_keputran_clean['harga'].notna()

mask_beras = data_keputran_clean['komoditas'] == 'Beras Medium'
awal = data_keputran_clean.loc[mask_beras & data_keputran_clean['harga'].notna(), 'tanggal'].min()
data_keputran_clean = data_keputran_clean[~(mask_beras & (data_keputran_clean['tanggal'] < awal))].reset_index(drop=True)

In [25]:
data_keputran_clean.head()

,komoditas_id,tanggal,komoditas,grup,satuan,harga_kemarin,harga,hari_nama,is_weekend,is_libur_nasional,nama_libur,is_ramadan,is_pra_ramadan,imputed
0,2,2020-01-01,Beras Premium,BERAS,kg,11000.0,11000.0,Rabu,0,1,New Year's Day,0,0,False
1,2,2020-01-02,Beras Premium,BERAS,kg,11000.0,11000.0,Kamis,0,0,NaN,0,0,False
2,2,2020-01-03,Beras Premium,BERAS,kg,11000.0,11000.0,Jumat,0,0,NaN,0,0,False
3,2,2020-01-04,Beras Premium,BERAS,kg,11000.0,11000.0,Sabtu,1,0,NaN,0,0,False
4,2,2020-01-05,Beras Premium,BERAS,kg,11000.0,11000.0,Minggu,1,0,NaN,0,0,False


In [26]:
data_keputran_clean.info()
data_keputran_clean[data_keputran_clean['harga'].isna()].groupby('komoditas')['tanggal'].count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67916 entries, 0 to 67915
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   komoditas_id       67916 non-null  int64         
 1   tanggal            67916 non-null  datetime64[ns]
 2   komoditas          67916 non-null  object        
 3   grup               67916 non-null  object        
 4   satuan             67916 non-null  object        
 5   harga_kemarin      66601 non-null  float64       
 6   harga              66765 non-null  float64       
 7   hari_nama          67916 non-null  object        
 8   is_weekend         67916 non-null  int64         
 9   is_libur_nasional  67916 non-null  int64         
 10  nama_libur         4605 non-null   object        
 11  is_ramadan         67916 non-null  int64         
 12  is_pra_ramadan     67916 non-null  int64         
 13  imputed            67916 non-null  bool          
dtypes: boo

komoditas
Beras Medium                       921
Garam Beryodium Halus               34
Kedelai Impor                       29
Minyak Goreng Curah                  4
Minyak Goreng Kemasan Premium      132
Susu Kental Manis Merk Indomilk      2
Terigu Protein Sedang (Kemasan)     29
Name: tanggal, dtype: int64

## 7. Save ke Data Processed

In [27]:
data_pucang_clean.to_csv('../data/processed/data_pucang_anom_clean.csv', index=False)
data_tambahrejo_clean.to_csv('../data/processed/data_tambahrejo_clean.csv', index=False)
data_wonokromo_clean.to_csv('../data/processed/data_wonokromo_clean.csv', index=False)
data_soponyono_clean.to_csv('../data/processed/data_soponyono_clean.csv', index=False)
data_keputran_clean.to_csv('../data/processed/data_keputran_clean.csv', index=False)

for nama, dfx in [('pucang_anom', data_pucang_clean), ('tambahrejo', data_tambahrejo_clean),
                  ('wonokromo', data_wonokromo_clean), ('soponyono', data_soponyono_clean),
                  ('keputran', data_keputran_clean)]:
    print(f"{nama:12s}: {len(dfx):,} baris | {dfx['komoditas_id'].nunique()} komoditas "
          f"| imputed {dfx['imputed'].mean()*100:.1f}%")

pucang_anom : 90,095 baris | 37 komoditas | imputed 0.1%
tambahrejo  : 77,920 baris | 32 komoditas | imputed 0.2%
wonokromo   : 77,920 baris | 32 komoditas | imputed 0.1%
soponyono   : 46,865 baris | 35 komoditas | imputed 0.1%
keputran    : 67,916 baris | 28 komoditas | imputed 0.2%
